In [1]:
import os
import sys
import global_functions as gl
import pandas as pd
import numpy as np
import plotly.express as px


### *Fatura Datalarının Temizliği Ve Anlamlandırılması*
> Tüm 12 KG Ev Tüpü Faturaları çekiliyor. 
Yurtiçi indirimde tüm 12 Kg ler üzerinden hesaplama yapıyorum 



In [2]:
sql_inc = """
SELECT 
[vbeln] as [talep_no],
[fkdat] as [tarihfatura],
[kunrg] as [kunnr],
[matnr] as [mamulno],
[fkimgton] as [miktar_ton],
[fkimg] as [adet],
[rafsatfyt],
[epdkkatpay] as [epdk],
[otv] as [otv],
[perlistfyt],
[hesmrj],
[hesnav],
[baydgtpay],
[aygpay],
[WADAT_IST] as [fiilimalcıkıs]
FROM [sqldb-aygaz-prod-001].[ods].[zaygbw_tup_ortfy]
WHERE matnr = 000000000006040012
AND fkdat > '2026-02-29' 
"""

df_new = gl.read_from_sql_multi(sql_inc, "Managed_Instance_BigDataUsr")
df_new["tarihfatura"] = pd.to_datetime(df_new["tarihfatura"], errors="coerce")


In [3]:
fatura=df_new.copy()

In [4]:
fatura['fiilimalcıkıs'] = pd.to_datetime(fatura['fiilimalcıkıs'], errors='coerce')
fatura.drop(fatura[fatura["hesnav"]==0].index, inplace=True)
fatura = fatura[fatura["fiilimalcıkıs"]> '2026-02-28']
fatura.sort_values(by=["fiilimalcıkıs"], ascending=True, inplace=True)
fatura["mamulno"]=fatura["mamulno"].astype(int)
fatura["kunnr"]=(fatura["kunnr"].astype(int)).astype(str)

In [19]:
fatura = fatura[fatura["fiilimalcıkıs"] < '2026-04-01']
fatura[fatura["kunnr"] == "1042075"]["miktar_ton"].sum()

np.float64(123.432)

In [18]:
fatura[fatura["kunnr"] == "1042075"]

,talep_no,tarihfatura,kunnr,mamulno,miktar_ton,adet,rafsatfyt,epdk,otv,perlistfyt,hesmrj,hesnav,baydgtpay,aygpay,fiilimalcıkıs
7711,3006403019,2026-03-03,1042075,6040012,4.992,416.0,23186.01,102.09,11383.00,1239.00,54899.75,4292.79,24597.03,30302.72,2026-03-03
7710,3006402824,2026-03-03,1042075,6040012,2.352,196.0,23186.01,102.09,11383.00,1239.00,54899.75,4292.79,24597.03,30302.72,2026-03-03
7709,3006402823,2026-03-03,1042075,6040012,0.048,4.0,23186.04,102.08,11382.92,1238.96,54899.79,4292.71,24597.08,30302.71,2026-03-03
7857,3006404411,2026-03-04,1042075,6040012,5.400,450.0,23186.01,102.09,11383.00,1239.00,54899.75,4292.79,24597.03,30302.72,2026-03-04
8077,3006405509,2026-03-05,1042075,6040012,5.460,455.0,23186.01,102.09,11383.00,1239.00,54899.75,4292.79,24597.03,30302.72,2026-03-05
9260,3006407097,2026-03-06,1042075,6040012,0.192,16.0,24118.75,102.08,10683.39,1249.01,55424.17,4292.81,24862.19,30561.98,2026-03-06
9380,3006409586,2026-03-09,1042075,6040012,3.240,270.0,24118.75,102.09,10683.40,1249.00,55424.18,4292.79,24862.18,30562.00,2026-03-09
9531,3006410531,2026-03-10,1042075,6040012,4.680,390.0,24118.75,102.09,10683.40,1249.00,55424.18,4292.79,24862.18,30562.00,2026-03-10
11515,3006411650,2026-03-11,1042075,6040012,7.680,640.0,24118.75,102.09,10683.40,1249.00,55424.18,4292.79,24862.18,30562.00,2026-03-11
11786,3006413589,2026-03-13,1042075,6040012,4.140,345.0,24118.75,102.09,10683.40,1249.00,55424.18,4292.79,24862.18,30562.00,2026-03-13


In [16]:
# Her kunnr için: Mart ayı toplam miktar_ton + son tarih satırındaki aygpay ve hesnav
mart_ton = (
    fatura[fatura["fiilimalcıkıs"].dt.month == 3]
    .groupby("kunnr")["miktar_ton"]
    .sum()
    .round(2)
    .rename("mart_miktar_ton")
)

son_satir_per_kunnr = (
    fatura.sort_values("fiilimalcıkıs")
    .groupby("kunnr")
    .last()[["aygpay", "hesnav"]]
)

fatura_ozet = mart_ton.to_frame().join(son_satir_per_kunnr, how="left")
fatura_ozet


,mart_miktar_ton,aygpay,hesnav
kunnr,,,
1001056,5.22,35721.63,5342.49
1001074,3.42,35721.63,5342.49
1001128,2.28,35721.63,5342.48
1001139,7.04,35721.63,5342.49
1001142,7.21,35721.63,5342.49
...,...,...,...
1441010,9.64,35809.55,5309.49
1446006,1.80,35094.18,5344.99
1450004,2.16,34913.14,4276.13


In [ ]:
fatura_ozet[.index fatura_ozet["kunnr"] == kunnr]

### Yurtiçi İndirim Ton Başı

In [88]:
ybd_indirim = pd.read_excel("/Users/testai/Desktop/price_analytics_cylinder/price_analysis_29-04-2026/yurtici_indirim.xlsx")

In [93]:
# Malzeme 6040012 için Ana hesap (kunnr) başına MRT 2026 değerleri
indirim_per_kunnr = (
    ybd_indirim[ybd_indirim["Malzeme"] == 6040012]
    .groupby("Ana hesap")[["MRT 2026"]]
    .first()
    .rename(columns={"Ana hesap": "kunnr"})
)
indirim_per_kunnr.index.name = "kunnr"
indirim_per_kunnr


,MRT 2026
kunnr,
1000178,0.0
1001056,-66154.0
1001074,-32766.0
1001128,-25509.0
1001139,-106000.0
...,...
1476004,NaN
1478008,-628320.0
1999111,-2488893.0


### Yurtiçi Fiili Nakliyat

In [94]:
ybd_fiil = pd.read_excel("/Users/testai/Desktop/price_analytics_cylinder/price_analysis_29-04-2026/fiilinakliye.xlsx",sheet_name="Mart 2026")

In [96]:
ybd_fiil=ybd_fiil[(ybd_fiil["Yıl"]==2026) & (ybd_fiil["Ay"]==3)][["OdeyenKodu","TL/TON"]]
ybd_fiil.rename(columns={"OdeyenKodu": "kunnr", "TL/TON": "fiil_nakliye_tl_ton"}, inplace=True)
ybd_fiil.set_index("kunnr", inplace=True)

In [97]:
ybd_fiil

,fiil_nakliye_tl_ton
kunnr,
1010039,3188.205793
1017017,4302.020104
1017038,7934.035208
1017041,6550.607190
1022013,6225.325810
...,...
1436006,10314.340327
1436007,16228.332043
1436008,11552.105585


### Fob Bayileri

In [6]:
ybd_fob = pd.read_excel("/Users/testai/Desktop/price_analytics_cylinder/price_analysis_29-04-2026/fobbayiler.xlsx")

In [9]:

ybd_fob["fob_tutar"] = np.where(ybd_fob["KM"] == 0, 0, (ybd_fob["KM"]+25)*(2*19.51) / 370)


In [10]:
ybd_fob

,Bayii Kodu,Bayii Tanımı,KM,TDM,fob_tutar
0,1001139,Hasan Akalın-Akalın Kardeşler Tic,0.000,Adana TDM,0.000000
1,1001142,Aziz Karut - Ezgi Ticaret,0.000,Adana TDM,0.000000
2,1001145,Abdullah Kaplan - Kaplan Ticaret,0.000,Adana TDM,0.000000
3,1001160,Sezikli Lpg Tüpgaz Maddeleri Paz.,0.000,Adana TDM,0.000000
4,1001174,Ay-Tüp Dağıtım Sanayi ve Ticaret,0.000,Adana TDM,0.000000
...,...,...,...,...,...
272,1455002,Altın Möble Ltd.Şti.,10.454,Samsun Terminal,3.738960
273,1455003,Altun Day.Tük.Mal.Tic.Ltd.Şti.,48.766,Samsun Terminal,7.779322
274,1461010,Ziya Hurşit Köse,0.000,Trabzon TDM,0.000000
275,1478008,Aydın Saray,315.204,Yarımca Terminal,35.877730


In [11]:
ybd_fob=ybd_fob[["Bayii Kodu", "fob_tutar"]].rename(columns={"Bayii Kodu": "kunnr"}).set_index("kunnr")
ybd_fob

,fob_tutar
kunnr,
1001139,0.000000
1001142,0.000000
1001145,0.000000
1001160,0.000000
1001174,0.000000
...,...
1455002,3.738960
1455003,7.779322
1461010,0.000000


### Tüm Bayiler Karlılık

In [12]:
# Tüm kunnr indexlerini string yap ve birleştir
fatura_ozet.index = fatura_ozet.index.astype(str)
indirim_per_kunnr.index = indirim_per_kunnr.index.astype(str)
ybd_fiil.index = ybd_fiil.index.astype(str)
ybd_fob.index = ybd_fob.index.astype(str)

# Tüm bayiler için kunnr bazında birleştirme
tum_bayiler = fatura_ozet.join([indirim_per_kunnr, ybd_fiil, ybd_fob], how="left")
tum_bayiler


NameError: name 'fatura_ozet' is not defined

In [13]:
tum_bayiler["ton_bası_karlılık"] = tum_bayiler["aygpay"] + tum_bayiler["hesnav"] - tum_bayiler["fiil_nakliye_tl_ton"].fillna(0) + (tum_bayiler["MRT 2026"]/tum_bayiler["mart_miktar_ton"])
tum_bayiler

NameError: name 'tum_bayiler' is not defined

In [14]:

tum_bayiler.reset_index().to_csv("/Users/testai/Desktop/price_analytics_cylinder/price_analysis_29-04-2026/tum_bayiler_karlılık.csv", index=False)
print("CSV kaydedildi: tum_bayiler_karlılık.csv")

NameError: name 'tum_bayiler' is not defined